In [ ]:
import os
import sys

sys.path.append("../src" if os.path.exists("../src") else "src")
import os
import pandas as pd
import matplotlib.pyplot as plt
from utils import REPORT_DIR, get_connection

In [ ]:
conn = get_connection()
df = pd.read_sql_query("SELECT * FROM delivery_master", conn)
os.makedirs(REPORT_DIR, exist_ok=True)

done = df.dropna(subset=["delay_hours"])
print(df.describe())

In [ ]:
def save_chart(data, kind, title, filename):
    """Draw one pandas chart and save it as reports/<filename>.png"""
    data.plot(kind=kind, title=title)
    plt.tight_layout()
    plt.savefig(REPORT_DIR + filename + ".png")
    plt.show()


def save_heatmap(table, title, filename):
    """Draw a coloured table with the numbers written inside the cells"""
    plt.figure(figsize=(7, 5))
    plt.imshow(table, cmap="coolwarm")
    plt.colorbar()
    plt.xticks(range(len(table.columns)), table.columns, rotation=45)
    plt.yticks(range(len(table.index)), table.index)
    for row in range(len(table.index)):
        for col in range(len(table.columns)):
            plt.text(col, row, round(table.iloc[row, col], 1), ha="center", va="center")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(REPORT_DIR + filename + ".png")
    plt.show()

In [ ]:
save_chart(df["status"].value_counts(), "bar", "Delivery status counts", "01_status")
save_chart(df["rating"].value_counts().sort_index(), "bar", "Rating distribution", "02_rating")
save_chart(done["delay_hours"], "hist", "Delay hours distribution", "03_delay_hist")

In [ ]:
save_chart(done.groupby("route_type")["delay_hours"].mean(), "bar", "Average delay by route type", "04_route")
save_chart(df.groupby("status")["rating"].mean(), "bar", "Average rating by delivery status", "05_rating_status")
save_chart(done.groupby("vehicle_type")["delay_hours"].mean(), "bar", "Average delay by vehicle type", "06_vehicle")
rating_by_level = df.groupby("experience_level")["rating"].mean()
rating_by_level = rating_by_level.reindex(["Junior", "Mid", "Senior", "Expert"])
save_chart(rating_by_level, "bar", "Average rating by driver experience", "07_experience")

In [ ]:
delay_table = done.pivot_table(index="route_type", columns="distance_category",
                               values="delay_hours", aggfunc="mean")
delay_table = delay_table[["Short", "Medium", "Long", "Very Long"]]
save_heatmap(delay_table, "Average delay: route type x distance category", "08_heatmap")

In [ ]:
numbers = df[["distance_km", "weight_kg", "experience", "driver_rating",
              "delay_hours", "delivery_hours", "rating"]]
save_heatmap(numbers.corr(), "Correlation matrix", "09_correlation")

In [ ]:
monthly_delay_rate = done.groupby("ship_month")["is_delayed"].mean() * 100
save_chart(monthly_delay_rate, "line", "Delay rate (%) by month", "10_monthly_delay")

seasonal_delay_rate = done.groupby("month_num")["is_delayed"].mean() * 100
save_chart(seasonal_delay_rate, "bar", "Delay rate (%) by calendar month (seasonality)", "11_seasonality")

weekday_delay_rate = done.groupby("day_of_week")["is_delayed"].mean() * 100
weekday_delay_rate = weekday_delay_rate.reindex(
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])
save_chart(weekday_delay_rate, "bar", "Delay rate (%) by day of week", "12_day_of_week")

print("12 charts saved in the reports folder")
conn.close()